In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

In [2]:
train_path = "/content/drive/MyDrive/train"
test_path = "/content/drive/MyDrive/test"
val_path = "/content/drive/MyDrive/val"

In [3]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

val_data = test_datagen.flow_from_directory(
    val_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

test_data = test_datagen.flow_from_directory(
    test_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

Found 2000 images belonging to 2 classes.
Found 16 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


In [4]:
model = models.Sequential([

    layers.Conv2D(32,(3,3),activation='relu',input_shape=(224,224,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),

    layers.Dense(128,activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(1,activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [5]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [6]:
history = model.fit(
    train_data,
    epochs=5,
    validation_data=val_data
)

Epoch 1/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 352s 5s/step - accuracy: 0.6690 - loss: 0.6618 - val_accuracy: 0.8125 - val_loss: 0.4236
Epoch 2/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 52s 830ms/step - accuracy: 0.8475 - loss: 0.3839 - val_accuracy: 0.8125 - val_loss: 0.3920
Epoch 3/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 53s 834ms/step - accuracy: 0.8790 - loss: 0.3166 - val_accuracy: 0.8125 - val_loss: 0.3995
Epoch 4/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 53s 847ms/step - accuracy: 0.8950 - loss: 0.2679 - val_accuracy: 0.8125 - val_loss: 0.4480
Epoch 5/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 53s 834ms/step - accuracy: 0.9055 - loss: 0.2509 - val_accuracy: 0.8125 - val_loss: 0.4106


In [7]:
test_loss, test_acc = model.evaluate(test_data)
print("Test Accuracy:", test_acc)

20/20 ━━━━━━━━━━━━━━━━━━━━ 176s 9s/step - accuracy: 0.8622 - loss: 0.3174
Test Accuracy: 0.8621794581413269


In [8]:
model.save("lung_disease_model.h5")

In [9]:
import numpy as np
from tensorflow.keras.preprocessing import image

img_path = "/content/drive/MyDrive/train/NORMAL/IM-0115-0001.jpeg"

img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img)/255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)


if prediction[0][0] > 0.5:
    print("Prediction: Pneumonia")
else:
    print("Prediction: Normal")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 942ms/step
Prediction: Normal
